# Catalog functions

Runs each `arora_data` catalog function against prod, read-only, as `readonly_user`. Change the arguments in any cell and re-run it.

In [ ]:
from IPython.display import Markdown

from arora_data.sync import list_columns, list_schemas, list_series, list_tables


def show(rows):
    rows = [r.model_dump() for r in rows]
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## `list_schemas()`
The data schemas and what each holds.

In [ ]:
show(list_schemas())

## `list_tables(schema)`
Each table's description, estimated rows and most recent date.

In [ ]:
show(list_tables("macro"))

In `market`, a blank (`None`) `latest_date` means the table is RLS-blocked until migration 136, or too big to read without a date index.

In [ ]:
show(list_tables("market"))

## `list_columns(schema, table)`
Column names and types. Most descriptions are `None`, because few columns have database comments.

In [ ]:
show(list_columns("macro", "observations"))

## `list_series(schema, table, limit)`
The series in a time-series table, with each one's name and most recent date.

In [ ]:
show(list_series("macro", "observations")[:10])

In `indicators.*`, `dataset` separates series that share a key, such as exports and imports here.

In [ ]:
show(list_series("indicators", "census_intltrade"))

Tables with no series list are refused before anything reaches the database.

In [ ]:
try:
    list_series("market", "price_history")
except ValueError as err:
    print("refused:", err)

A result bigger than `limit` raises `RowCapError` instead of coming back cut off.

In [ ]:
from arora_data._db import RowCapError

try:
    list_series("macro", "observations", limit=10)
except RowCapError as err:
    print("refused:", err)

## `to_df(rows, model)`
Any result as a Polars dataframe. The model sets the columns, so an empty result keeps them too. `.to_pandas()` also needs pandas and pyarrow installed.

In [ ]:
from arora_data.catalog import SeriesInfo, TableInfo
from arora_data.frames import to_df

to_df(list_series("macro", "observations"), SeriesInfo).head(10)

In [ ]:
to_df(list_tables("macro"), TableInfo)

In [ ]:
to_df([], SeriesInfo)